In [ ]:
from neo4j import GraphDatabase
import os
import json
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
load_dotenv()  # will read .env in current dir

In [ ]:
uri = os.getenv("NEO4J_URI_3")
user = os.getenv("NEO4J_USERNAME_3")
password = os.getenv("NEO4J_PASSWORD_3")
DB  = os.getenv("NEO4J_DATABASE", "neo4j")

OPENAI_API_KEY = os.environ["OPENAI_API_KEY"] 

driver = GraphDatabase.driver(uri, auth=(user, password))
print("[INFO] Connecting to:", uri)
print("[INFO] DB:", DB)
print("[INFO] USER:", user, "PWD:", password)
llm = ChatOpenAI(temperature=0, model_name="gpt-4o", api_key=OPENAI_API_KEY)

[INFO] Connecting to: neo4j+s://0f3e31f8.databases.neo4j.io
[INFO] DB: neo4j
[INFO] USER: neo4j PWD: HY8U1x-_cb19tGvL9S6LVfNVqliokWKY2F1tFiKCdec


In [ ]:
from langchain.embeddings import OpenAIEmbeddings
embedder = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)  # or your desired model



In [ ]:
# Function to load JSON data
def load_knowledge_graph(json_file_path):
    with open(json_file_path, 'r') as file:
        return json.load(file)

In [ ]:
def create_node(tx, node):
    # Initialize empty attributes dict if not present
    attributes = node.get("attributes", {})
    
    # Fallback: create 'name' from id (for meaningful embedding) if missing
    text_for_embedding = (
        attributes.get("name")
        or attributes.get("description")
        or node.get("id")
    )
    if text_for_embedding:
        embedding_vector = embedder.embed_query(text_for_embedding)
        attributes["embedding"] = embedding_vector

    # Original label (escaped & cleaned)
    raw_label = node["label"].replace(" ", "_").replace("-", "_")
    label = f"`{raw_label}`"

    # Build attributes map
    attributes_str = ", ".join([f"`{key}`: ${key}" for key in attributes.keys()])

    # 🔹 Add Embedded *and* specific label
    query = f"""
    MERGE (n:Embedded:{label} {{id: $id}})
    {"SET n += {" + attributes_str + "}" if attributes_str else ""}
    """
    tx.run(query, id=node["id"], **attributes)


In [ ]:
def create_relationship(tx, relationship):
    if "type" not in relationship or "source" not in relationship or "target" not in relationship:
        print(f"Skipping relationship due to missing fields: {relationship}")
        return

    attributes = relationship.get("attributes", {})  # Likely empty in your new format
    attributes_str = ", ".join([f"{key}: ${key}" for key in attributes.keys()])
    rel_type = f"`{relationship['type'].replace(' ', '_').replace('-', '_')}`"

    query = f"""
    MATCH (a {{id: $source}}), (b {{id: $target}})
    MERGE (a)-[r:{rel_type}]->(b)
    {"SET r += {" + attributes_str + "}" if attributes_str else ""}
    """
    tx.run(query, source=relationship["source"], target=relationship["target"], **attributes)


In [ ]:
def store_knowledge_graph(driver, graph):
    START_FROM_NODE_INDEX = 18565
    with driver.session() as session:
        print("[INFO] Storing Nodes...")
        total_nodes = len(graph["nodes"])
        for idx, node in enumerate(graph["nodes"][START_FROM_NODE_INDEX:], start=START_FROM_NODE_INDEX + 1):
            if "id" not in node or "label" not in node:
                print(f"[WARNING] Skipping node with missing 'id' or 'label': {node}")
                continue
            print(f"[INFO] Adding Node {idx}/{total_nodes}: ID = {node.get('id')}, Label = {node.get('label')}")
            session.write_transaction(create_node, node)

        print("[INFO] Storing Relationships...")
        for idx, relationship in enumerate(graph["relationships"], start=1):
            print(f"[INFO] Adding Relationship {idx}/{len(graph['relationships'])}: Type = {relationship.get('type')}, Source = {relationship.get('source')}, Target = {relationship.get('target')}")
            session.write_transaction(create_relationship, relationship)



In [ ]:
# Load the knowledge graph data from a JSON file
json_file_path = "/mnt/SAS_A/srushti_thesis/Final_Code/PoisonedRAG/input_jsons/25k_28k.json"  # Path to your JSON file
with open(json_file_path, "r") as file:
    knowledge_graph = json.load(file)

# Store the knowledge graph in Neo4j
try:
    store_knowledge_graph(driver, knowledge_graph)
    print("Knowledge graph stored in Neo4j successfully!")
finally:
    driver.close()

/tmp/ipykernel_3646474/987858751.py:3: DeprecationWarning: Using a driver after it has been closed is deprecated. Future versions of the driver will raise an error.
  with driver.session() as session:
/tmp/ipykernel_3646474/987858751.py:11: DeprecationWarning: write_transaction has been renamed to execute_write
  session.write_transaction(create_node, node)


[INFO] Storing Nodes...
[INFO] Adding Node 3103/18565: ID = black_panther, Label = work
[INFO] Adding Node 3104/18565: ID = dolby_theatre, Label = location
[INFO] Adding Node 3105/18565: ID = coogler, Label = person
[INFO] Adding Node 3106/18565: ID = dora_milaje, Label = other
[INFO] Adding Node 3107/18565: ID = doc25338, Label = document
[INFO] Adding Node 3108/18565: ID = world_premiere, Label = event
[INFO] Adding Node 3109/18565: ID = black_panther, Label = entertainment
[INFO] Adding Node 3110/18565: ID = riyadh, Label = location
[INFO] Adding Node 3111/18565: ID = saudi_arabia, Label = location
[INFO] Adding Node 3112/18565: ID = april_18_2018, Label = date
[INFO] Adding Node 3113/18565: ID = king_abdullah_financial_district, Label = location
[INFO] Adding Node 3114/18565: ID = amc_theatres, Label = organization
[INFO] Adding Node 3115/18565: ID = disney, Label = organization
[INFO] Adding Node 3116/18565: ID = italia_film, Label = organization
[INFO] Adding Node 3117/18565: ID 

/tmp/ipykernel_3646474/987858751.py:16: DeprecationWarning: write_transaction has been renamed to execute_write
  session.write_transaction(create_relationship, relationship)


[INFO] Adding Relationship 2/30256: Type = LOST_PLAYER, Source = maple_leafs, Target = world_hockey_association
[INFO] Adding Relationship 3/30256: Type = PARTICIPATED, Source = 1972_73_season, Target = maple_leafs
[INFO] Adding Relationship 4/30256: Type = DRAFTED, Source = lanny_mcdonald, Target = 1973_nhl_amateur_draft
[INFO] Adding Relationship 5/30256: Type = MANAGES, Source = jim_gregory, Target = maple_leafs
[INFO] Adding Relationship 6/30256: Type = TRADED_PICK, Source = philadelphia_flyers, Target = maple_leafs
[INFO] Adding Relationship 7/30256: Type = TRADED_PICK, Source = bruins, Target = maple_leafs
[INFO] Adding Relationship 8/30256: Type = DRAFTED, Source = bob_neely, Target = 1973_nhl_amateur_draft
[INFO] Adding Relationship 9/30256: Type = DRAFTED, Source = ian_turnbull, Target = 1973_nhl_amateur_draft
[INFO] Adding Relationship 10/30256: Type = ACQUIRED, Source = borje_salming, Target = maple_leafs
[INFO] Adding Relationship 11/30256: Type = MENTIONED_IN, Source = wor

ClientError: {code: Neo.ClientError.Transaction.TransactionHookFailed} {message: You have exceeded the logical size limit of 400000 relationships in your database (attempt to add 4 relationships would reach 400002 relationships). Please consider upgrading to the next tier.}

In [ ]:
Load the knowledge graph data from a JSON file
json_file_path = "/mnt/SAS_A/srushti_thesis/Final_Code/PoisonedRAG/input_jsons/20k_23k.json"  # Path to your JSON file
with open(json_file_path, "r") as file:
    knowledge_graph = json.load(file)

# Store the knowledge graph in Neo4j
try:
    store_knowledge_graph(driver, knowledge_graph)
    print("Knowledge graph stored in Neo4j successfully!")
finally:
    driver.close()

SyntaxError: invalid syntax (3306092957.py, line 1)

In [ ]:
def get_all_node_labels():
    with driver.session() as session:
        result = session.run("CALL db.labels()")
        return [record["label"] for record in result]

In [ ]:
# Example usage
labels = get_all_node_labels()
print("Node Labels:", labels)

/tmp/ipykernel_63519/522634782.py:2: DeprecationWarning: Using a driver after it has been closed is deprecated. Future versions of the driver will raise an error.
  with driver.session() as session:


Node Labels: ['other', 'organization', 'document', 'person', 'financial_term', 'work', 'date', 'keyword', 'law', 'location', 'event', 'keyword_label', 'scientific_term', 'product', 'language', 'title', 'concept', 'character', 'award', 'project', 'facility', 'treatment', 'music', 'economic_term', 'economic_policy', 'policy', 'financial_instrument', 'activity']
